In [1]:
!pip install llama-cpp-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 129.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.9-cp311-cp311-macosx_15_0_arm64.whl size=3472051 sha256=faaeafa774464e034d57aa731c03916f01e65ac1109ab5e12b4d5d0b499e82d0
  Stored in directory: /Users/baisakhisarkar/Library/Caches/pip/wheels/9e/8f/bf/148c8eb7d69021eccd6eae6444f3accd48347587054ffd24e5
Successfully built llama-cpp-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [3]:
import pandas as pd
from llama_cpp import Llama

In [4]:
excel_path = "data/lsst_forum_responses_5.xlsx"
df = pd.read_excel(excel_path)

questions = df['question'].dropna().tolist()

In [5]:
# GGUF model paths and display names
model_configs = [
    {"path": "gguf_models/OLMo-2-0425-1B-Instruct-Q4_K_M.gguf", "name": "OLMo-2-0425-1B-Instruct-Q4_K_M"},
    # Add more like this
    # {"path": "other_model.gguf", "name": "Other-Model"},
]

In [6]:
# Helper to load GGUF models
def load_llama_model(model_path):
    return Llama(
        model_path=model_path,
        n_ctx=2048,
        n_gpu_layers=0  # Use CPU; change if using GPU
    )


In [ ]:

def generate_from_gguf_models(questions, model_configs, max_tokens=200, temperature=0.7):
    results = {"question": questions}
    
    for model in model_configs:
        print(f"Loading model: {model['name']}")
        llm = load_llama_model(model['path'])

        model_outputs = []
        for q in questions:
            prompt = f"Q: {q.strip()} A:"
            response = llm.create_completion(
                prompt=prompt,
                max_tokens=max_tokens,
                temperature=temperature
            )
            answer = response["choices"][0]["text"].strip()
            model_outputs.append(answer)

        results[model["name"]] = model_outputs

    return pd.DataFrame(results)


In [8]:
final_df = generate_from_gguf_models(questions, model_configs)
final_df.to_csv("gguf_model_responses.csv", index=False)

final_df.head()


llama_model_load_from_file_impl: using device Metal (Apple M2 Pro) - 10916 MiB free
llama_model_loader: loaded meta data with 42 key-value pairs and 179 tensors from gguf_models/OLMo-2-0425-1B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = olmo2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = OLMo 2 0425 1B Instruct
llama_model_loader: - kv   3:                             general.author str              = AllenAI
llama_model_loader: - kv   4:                       general.organization str              = AllenAI
llama_model_loader: - kv   5:                           general.finetune str              = Instruct
llama_model_loader: - kv   6:                   

Loading model: OLMo-2-0425-1B-Instruct-Q4_K_M


load_tensors: offloading 0 repeating layers to GPU
load_tensors: offloaded 0/17 layers to GPU
load_tensors:   CPU_Mapped model buffer size =   888.79 MiB
.....................................................................
llama_context: constructing llama_context
llama_context: n_seq_max     = 1
llama_context: n_ctx         = 2048
llama_context: n_ctx_per_seq = 2048
llama_context: n_batch       = 512
llama_context: n_ubatch      = 512
llama_context: causal_attn   = 1
llama_context: flash_attn    = 0
llama_context: freq_base     = 500000.0
llama_context: freq_scale    = 1
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
ggml_metal_init: allocating
ggml_metal_init: found device: Apple M2 Pro
ggml_metal_init: picking default device: Apple M2 Pro
ggml_metal_init: GPU name:   Apple M2 Pro
ggml_metal_init: GPU family: MTLGPUFamilyApple8  (1008)
ggml_metal_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_init: GPU family:

,question,OLMo-2-0425-1B-Instruct-Q4_K_M
0,"Hello, Rubin website team- \n This is to repor...","This is a minor issue, and I believe it has be..."
1,"Hello, Rubin website team- \n This is to repor...",Rubin Scientist Outreach Program\n\n---\n\n**U...
2,"Hello, Rubin website team- \n This is to repor...",Technical Support Engineer at Rubin \n\nHere a...
3,"Date: Thu Feb 6 2025, 9:00-10:00am PST ( 2025...",Rubin’s 1st Data Assembly meeting; B: Rubin’s ...
4,Apologies for short notice. Due to planned mai...,I understand. We appreciate your patience. Ple...
